In [2]:
!pip install -q langchain langchain-classic langchain-community langchain-groq \
    langchain-chroma langchain-huggingface langchain-text-splitters \
    sentence-transformers pypdf

In [3]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [4]:
#download a pdf
!wget -q -O my_paper.pdf https://arxiv.org/pdf/1706.03762
!ls -lh my_paper.pdf

-rw-r--r-- 1 root root 2.2M Apr 12  2024 my_paper.pdf


In [5]:
#load the pdf
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("my_paper.pdf")
data = loader.load()

print(len(data))                  # one Document per page
print(data[0].metadata)           # {'source': 'my_paper.pdf', 'page': 0, ...}
print(data[0].page_content[:300])

/tmp/ipykernel_788/1290607323.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


15
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'my_paper.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [6]:
#chunk
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
docs = text_splitter.split_documents(data)

print("Total chunks:", len(docs))    # roughly 40-60 for a 15-page paper
print(docs[3].page_content)

Total chunks: 52
efficient inference and visualizations. Lukasz and Aidan spent countless long days designing various parts of and
implementing tensor2tensor, replacing our earlier codebase, greatly improving results and massively accelerating
our research.
†Work performed while at Google Brain.
‡Work performed while at Google Research.
31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.
arXiv:1706.03762v7  [cs.CL]  2 Aug 2023


In [7]:
#Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector = embeddings.embed_query("hello, world!")
print(len(vector))      # 384
print(vector[:5])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384
[-0.03817715868353844, 0.03291110321879387, -0.005459390580654144, 0.014369942247867584, -0.040291037410497665]


In [8]:
#vector store
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)
print("Stored chunks:", vectorstore._collection.count())

Stored chunks: 52


In [9]:
#Retriever with k=10
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 10})

question = "What is the main contribution of this paper?"
retrieved_docs = retriever.invoke(question)

print(len(retrieved_docs))   # 10
for i, d in enumerate(retrieved_docs):
    print(f"--- chunk {i} | page {d.metadata['page']}")
    print(d.page_content[:200])

10
--- chunk 0 | page 11
[25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated
corpus of english: The penn treebank. Computational linguistics, 19(2):313–330, 1993.
[26] David McCl
--- chunk 1 | page 0
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need

--- chunk 2 | page 2
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as 
--- chunk 3 | page 12
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS
--- chunk 4 | page 3
dot product attention without scaling for larger values of dk [3]. We suspect that for

In [10]:
#LLM(Groq)
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3,
    max_tokens=1500,
)

In [11]:
#prompt and chain
from langchain_core.prompts import ChatPromptTemplate

try:
    from langchain.chains import create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except ImportError:
    from langchain_classic.chains import create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Answer ONLY using the retrieved context below. "
    "Do not use any outside knowledge, even if you know the answer. "
    "If the answer is not in the context, reply exactly: "
    "\"I don't know based on the provided document.\" "
    "Use three sentences maximum.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [12]:
#ask question
response = rag_chain.invoke({"input": "What is the main contribution of this paper?"})
print(response["answer"])

The paper introduces the Transformer, a new neural architecture that relies solely on attention mechanisms and removes both recurrence and convolutional layers. It proposes scaled dot‑product and multi‑head attention as the core building blocks. The Transformer achieves state‑of‑the‑art performance on sequence transduction tasks such as machine translation.


In [13]:
#show the pages used
for d in response["context"]:
    print("page", d.metadata["page"] + 1)

page 12
page 1
page 3
page 13
page 4
page 11
page 4
page 11
page 9
page 15


In [14]:
#don't know test
print(rag_chain.invoke({"input": "Who won the 2022 football world cup?"})["answer"])

I don't know based on the provided document.


In [16]:
%%writefile pdf_app.py
import hashlib
import tempfile
import streamlit as st
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
try:
    from langchain.chains import create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except ImportError:
    from langchain_classic.chains import create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

st.title("PDF Q&A with RAG (Groq)")

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Answer ONLY using the retrieved context below. "
    "Do not use any outside knowledge, even if you know the answer. "
    "If the answer is not in the context, reply exactly: "
    "\"I don't know based on the provided document.\" "
    "Use three sentences maximum.\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{input}")])

@st.cache_resource
def get_embeddings():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

@st.cache_resource
def build_chain(pdf_bytes: bytes):
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as f:
        f.write(pdf_bytes)
        path = f.name
    data = PyPDFLoader(path).load()
    docs = RecursiveCharacterTextSplitter(chunk_size=1000).split_documents(data)
    vs = Chroma.from_documents(
        documents=docs,
        embedding=get_embeddings(),
        collection_name="pdf_" + hashlib.md5(pdf_bytes).hexdigest()[:12],
    )
    retriever = vs.as_retriever(search_type="similarity", search_kwargs={"k": 10})
    llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3, max_tokens=1500)
    qa = create_stuff_documents_chain(llm, prompt)
    return create_retrieval_chain(retriever, qa), len(data), len(docs)

uploaded = st.file_uploader("Upload a PDF", type="pdf")

if uploaded:
    rag_chain, n_pages, n_chunks = build_chain(uploaded.getvalue())
    st.caption(f"{n_pages} pages -> {n_chunks} chunks")

    query = st.chat_input("Ask a question about the PDF: ")
    if query:
        st.chat_message("user").write(query)
        response = rag_chain.invoke({"input": query})
        st.chat_message("assistant").write(response["answer"])
        with st.expander("Sources used"):
            pages = sorted({d.metadata.get("page", 0) + 1 for d in response["context"]})
            st.write("Pages:", pages)
else:
    st.info("Upload a PDF to begin.")

Overwriting pdf_app.py


In [17]:
!pip install -q streamlit
import subprocess, os, time
subprocess.Popen(["streamlit", "run", "pdf_app.py", "--server.port", "8501",
                  "--server.headless", "true"],
                 stdout=open("/content/logs.txt", "w"), stderr=subprocess.STDOUT,
                 env=os.environ.copy())
time.sleep(8)
!curl -s ipv4.icanhazip.com
!npx localtunnel --port 8501

136.85.54.194
⠙⠹⠸⠼Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦your url is: https://cold-laws-clap.loca.lt
fifa world cup
^C
